# Lab 2 · Light, colour and histograms
**Computer Vision · Session 2 · Easy–medium · 120 minutes**

How can two photographs of the same scene have different pixel values—and when can we improve them?

**Work pattern:** run the example → predict → complete a few similar lines → explain your evidence. Write answers in the text cells. The goal is computer vision reasoning, not building a program.

Your student ID assigns two dataset photographs and numerical settings. Keep these settings in your submission. Different IDs can share a photo, but receive different combinations of settings. All tasks are independent after the two setup cells: an unfinished task does not block the next section.

| Section | Session 2 concepts | Time |
|---|---|---:|
| Setup | Your images and assignment | 5 min |
| 1 | Light, optics, sensor, sampling, perception | 15 min |
| 2 | RGB, luminance, HSV, white balance | 20 min |
| 3 | Histograms and lost spatial information | 15 min |
| 4 | Affine contrast, gamma, clipping | 20 min |
| 5 | Stretching, CDF, equalisation, CLAHE | 25 min |
| 6 | Independent transfer and conclusion | 15 min |
| Preview | Filtering and edges next week | 5 min |

This lab assumes Lab 0 skills: running cells and working with image arrays. It does not repeat image-loading exercises or the Lab 1 classifier.

## Setup · Run once
The small photo dataset is included in the notebook; no download, account or GPU is needed. Expand the setup only if you are curious.

In [ ]:
#@title Load the dataset and display helpers (run once) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
from PIL import Image
import cv2, hashlib, base64, io
PHOTO_DATA = {'astronaut': '...', 'coffee': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAkGBwgHBgkIBwgKCgkLDRYPDQwMDRsUFRAWIB0iIiAdHx8kKDQsJCYxJx8fLT0tMTU3Ojo6Iys/RD84QzQ5Ojf/2wBDAQoKCg0MDRoPDxo3JR8lNzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzf/wAARCADIASwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDxsVMo4pirzTyewrNmiHZ7U4N2pF4FBpDHZpQc0zNKoOaQx9KBSqMkDqa1bDS2Z0a4Xr0j7n61E5qKuzSnSlUdolWzsZboggbUz94/0rcsdOVW2wjAB5c1oWtmWO1iqheNo9K0FiVEAO35e3QCvPq4hy0R69HCxpavVkdpZRRsS/JJ5J7/AEqxMmxW3MFBHyqD0+tN88sdsSAe5GSf8KY5SLBYb5D0AGa51ds6HpuO8ouAXY7R69T/AIVG86xkLGDwOw/lTJbkIWaQbnx9zPA+tU5BNdyFg55HODjAq1EhtsWcsX5BQHkY5Jp8diZELTfIpOcdDW9oPhPUb3bJFC+ztNN8q/h3NdhZeAIBh9QvHkP92MBF/M81tGlUl8KMJ4inD4mefLKsahIlUKP4RTTIZjkjjPWvWIPCfh62/wCXSNz6yMzH+dWhpOhKMfYLXH/XFav6o+rRj9eh0izyAgZyTnj8qcYyQDuAFeuPougyrtaxtse0YH8qp3Hg3Q7jPlI0R/2JCP0OaTwcujQ1j6fVNHmEUBA3EfL/ADp7AgZPA6V21/4CmX57K7VwOkcq4/UVy+p6Xe2DgXlu8Q6bzyp+hHFc9SjUh8SOqniKdT4WVIUVDvdwD6kdKe7uCTyxHGRUDYZCpAwBnJ7fWkSXdtSMnBHLeo/oKy3NbEx29W5I5x2H+NBJ4Y5NBwFCAdOtHlkrhBTSE2PX58ggYqRsL1OKhEbRgYPHc1GZt2R29TTFa5Z3BvpQjqW285qKJWwcd+lTJESmThTnmhCaRKg8s8fMfU1LCxd8E8jue1QxqSRuOB2qdgoGQQMelNEtdx7JtPyjJqeKcBCmfr71T884Cjp604KZMbOWHYVokZSLW5hgjpmr9tEswBb73as1DsI8zjPG7tWjBMsPMuRjp71aRjJl2ODCFWABB+8aYJyvCqSB3pZbjeq7z14AFNSF5BuaQp6KB0puxn6nziDil6mkBoJr0zzh+KMHNNUnNOzQMM4qe2hluZAkS5P8qtWWmSz7XkBRD69TXT2enxRhFjAQDsOrGuariIw0W524fByqay0RT0zS1t2+b5pT3xnH0rajRVIHlEE8cdW+lWFWNQwVlB6fL/jSbGkIwxCkYJHVvp7V5s6jm7s9eFONNWihGQBhjbkdQOg+p9aApkOS21P72P5VIyJEMSbSB0QHj6k1E0jSg8be47cf0FJLqDl2FlmS2xCi7SeSTyT9f8KgM6rKBk+YfbLY/oPag7N58pdzEcv/AHfpV/StKlnnSNEzLK2FXufr6CtLIyb7kOmaTLqk6xwqX3H5VUdfc+gr0vQvCNlpyJNeIk9wOcY+RPoO/wBTWjoekW+j2oRApmYfvJAOp9B7VBq+qiAFEOW/lXZGEaa5pbnnVa8qr5YaI0LrUIrZeWAx0FY9xrUkhPl8D1NYklw8zlnJJoWQelRKs2KNFI0jeTSHlzQJXPVjVJZB71Mjg9GqVIrlLiyuP4jU0d3IvfNVFf1qRSD0NWmQ0atvqZDAElRV8TwXUbRzKrK3B4yD9RXOipI5GjOVNaxqNGUqaexU8Q+CIbhPO0shCOfIJ+Rvp6H68Vw09pLbSGOaMxSIfmBGCK9Ys77JwcD1HY1W8R6Bb63a7kIjuVHySf0PqKzq4eNRc0NGdFDFypvlqarueXx5fIAOe9TFowfvNj0z1omhktJnt5wY5EO1lNVpFfzeOlefZnqaMszbXXklVHpVZI2bO7GB0FTqQseCM0iKVfB79TimCFjGAOmasAbvvN09OlMMZXnHB7dz/wDWpjYPy547CkgZa3Atj/JqJ9xOD90VUVSrEq5B9M9KsK6kDeccc5rVIxbHiIzEsD9BU8QaF1Cnr3NV1lH8HSp4pBI2OoHerMmXlaOVGjbhu3pTZlkiVc/Oy8qtRo6q4RRznP0qQKHmZmYs2eSepobJ5balq2UvOJ5SSc/KvpWuZg5LED8OKzYsjBx2xxU2cEgggg4IprQiWp86Y4po61Iw4q5p+myXH7xgUh7tj+VenKSirs86EJTdoorwW8kzBIULuewrf0zQkjCyXhG4ngdf0rT060FqvlWsIV2+9I3Lbavwood2VCxGfnxwfYV59XEt6R2PVoYOMNZ6sRbdVTdvKqOMgDmpldMbYY856+/1NNgAlk+cM5HTPAq7kJkkYPQYGPwA7CuNtnfoNt7UAs8mPcdABRPeJAwjhUu7cDAyzfT2qC4v/LbygBv+mQo/qahiuVRj5SHe4wXk6n/PpTUXuyJS7CSGTfvuAoB5Cg9KgaUylY0UcjIQH9TT0VnjaeaTEWcAsOWPsO9MYbBz+7X+7n5m+prS6QQpyn6GhawtDt8uMzTnkInQe5/x6V6D4C0iWGGTU71o3mm+WMIchF78+pP8q8zjup7qSKygxGksirsTjJJxyep/Gvc7S3SzsobeIYSJAoH0Fb4aN5OT6HLj37KCgupW1a8FtATnntXE3E7zSMxOSa1/ElyWn8sHgVhCpqz5pHNRhaI8Egck0bveoy1RtKBWNzexY8wjuacLhh3/ADqgZsf4Cmm5UHkj86XOPkNeO9Ydf0q3DeIxGTg1gJcqehH51OkoPeqVQl0zpY5Qw6/jUtc9DcPGco34Vp2t8r4V/lNbxqJmMoNF9WKnIrW0+63cE1jnmn28pjlFaxlZ3MZRuhvjTQ1voFv7dR50RAk/2k9fqK5a20yNIz54U7s4J5r0uBhLHhhlWGCPauVurIRTyQFf9U3y+uKqrCLfOluKnVmlyX2ObvbCOBIbm3l3Qy5XaeqOOo9+o5qqGAY7scd+w/xNXtVljEyQRgLtBJHfJqkAFOW9OOK86pbm0PWo3cE2IzHGEUsTUMqEHAAz3IqRnJIx8oqKSXauFPbvSimUyAxhMlySx7A9qXY7cHlc4FIpyCWOTT4mbb7kcA1qmZtCoPukA8cGrMTsF4Hy5/KkiVWj2qepxmrCJ8u0jHHai9xWsKryHb/d6e4qYuMErwf5mmRREgqWBGOo9afDGGfpke/amtDN6l63ckAuccVoRrHIu5zICfQVWjRYlWSQZB4K/wBasKoI+Ujb257U0ZyPD9O0cj97cJuP8Kds11NlpUkygg+VGMZB6L9ataZYRQKHlJkcnPA6n29KtXVyqHBXcF/h/gX/AB/z1rKrWlNnbTpwpRtFFWVI7VUjiZjGRyf4pj/RaZc3G4RxxIFDDsO39KiSOS6nL/ebHzysflX2/wDrUtxcR2wxGxeRuPMPU/T0FJIpj0UW2395hwc7RzimXEsm7ewCR5yW/wAB61TeZcnAD45OeAaiXzruY5wdvQY4H/1q0Ue5m27joVQbpn3YOdgPJalWLyUV5cMW5A9auSRR2qtNOd8p6ZHX2qjK7+YXlOXP6UOV9jWjR5ndku8mQM5y2OPRarSOXYknNMZyxp4wy4A+apsdmxpeFVD+JdNDdPtC17q4yjEdq8C0a5FlrNjcPwI7hCfYZ5r39DujYDutd2G1i0eJmt/aRfkefau5a9lJ/vVQZgBVzWAY7+dT2Y1zOsX2zMMbYJ+9jr9K4JvU0pQ5rJDrnWVwwgU5H8TcCsqfWJudrAZ7gVTkbIO4Zz0Ap1tYyTEbvlXucZNKy3kdXLFbIjfUZT1duPeo/trk5Jb8q24NMtkH/Hu8zesj7R+Q/wAatLaRAcadaY9w5/rRz00GpzqXxB4Yir1tqMi4Ic/nWqunWEh/0mw2j+9A5BH4EkUyTwxDOc6XdZftDPhWP0PQ0XhLYTlbcls9XyQJRkeorZhmWRQyNkGuMlt7izmMVxG0cinBVhir1hePC4weO4pXcSZUlJXR3NjelSI5TlT0PpWgzYIIrmreZZUDKeDWvaT74trHkV0U53OKcLHWaY+6EVT1eNWvJG6bogSfpnNT6ISbfNc18S9S+wWEgR2WSaExJg92YD+W6utv90ckIc1XlRzNxLiaSaYFXJyVYdPT9KjWZnYdw361JpWpQ67bx2N7tW+RQsEx4EwHRGPY+hqB4XtZWR1ORkAHggjt7EV5rWup7bvHTsTShQODz6VTl4/iOO9SNKAp55NKIwTgkH0p7EX7laNd752lU7mr4jCoo7enc/WiNApyACQOOKaXJYYyT0+lF7gSBljOXxuHGPSnFmcAjIquqMZCzdPrVpWHy5GB/nmnexLVy1bbtwOPqKuWxDkhcdOhqlbyNJLhcnHetDymDLgDnnK9ad7mUlYc0jogUgnB71E90inB3A+1XYQrKjK25geVZarTxWYk+YkEjoOcVaRm3qcJLeOzctt+XCqvXHvTzG3lrNfuUhPKoDhmoZodM6hLi7xnaPuJ/jWfI8l3K0t1Izk8g9gfTFZqJ23ZYuL5pFCIpjhH3I171WVnkYYwXY49dvtViKzklYlWKx95GFalnYQwgOQWb+83BNVoiW7Gdaac3mA3GduclfetTYkEZIXAHPJ4/H1qw8UcZ5OTjgYNZGrSmNfLzukkxhhwAPQUm2xQXM7EF7cC5dZeCFGAuMDNZ7tk1K7BWXjIHUetF7MlxO0kcYjB/hXoKEj0EuVWRBTgSDkGmilpjJNQuDcICUVCAB8v869w8GasNW8P2lyT+9CbJR6OvBrwicZhfHpXV/DnxH/ZOri1unxZ6jhkYnhJeh/P/CuihLlZ5uYUeeGnQ7DxzAbSU3ij5XXH/Aq85eN55ScEsxr2/WNOh1bTpLaX7rj5W/unsa83OkSWNxJDOmHU4+o9a58XF05c3RmWBqxlDle6Me00tRhpBuNakVoB/DirscGO1WFiA7V50ptna2Ukth6VKLcelWwuO1IRU3JuVfIHpTGthVyggUXFciMcV/ELXUACRxFOfvJ7E9xXP3elvaztGRypxXRsuaber56IzcsowT61qqjsEdHoY2mu0UnlN0bpW5asQ496y5odoDjgqc10Xh+xN7dIxH7pPmY/0reg3J2RjiLJczOs0qLyrOMNwcZNeTfFTUBd+JobJDlbaMF/97k/+zV6trOowaRpc95cMFjhQsff0FfPcl1LqOp3N7OcyzOWb2J5x+HT8K9Oq+WKicuX03Oq5snjyjAqSP6V2dheprVkwuOb2BP3rDrKg4D/AO8vf1H0rjRwKsWV1LZ3MdxA22SNsg/0rhZ7c6fOvM1po5IbgQld0gP4Y9atxA4Azj1/+tVuVYb21S6tgQgTcAOqrnBX/gLHH0YVVk/ebQn0NJnDrfUdvAO2Pr3p8cSb1zkN6ioVGG2rww7+lTq2z8Dmi5JO0cZjAHJHTApiQGV8EnaOrU9PnHmgbV9M9qleTYACBnsBTsRzEhX7NGPLwVJ5IqeGd/KeQI7Kg+bHbPc1n3EhEOQd2DyM4zVUXciq4R3+fqM8EelUkkRqzdM5mMNxblE2rskQ53Z9c1IYGU5VsBueawEvJQwjBUK3Qg1ox3aqgDZz35q0Ztdjio4RMxIBx3xzn6mrtrYZGGQhfTOS3+Aq/b2KxRqqLhey9z7/AP1/yq8EjgQsx2gDBwOef8+/49azOuU+xTWFRhWIAQdgMD2/z+tJ58ZVUgDszcFiuD+H+fzp825zn5lXsCfzP+f/AK9RHGQFDZbjGMk/jRYi9xXkcsUTBz1A5H0rn7uQvfSMTnZwK6C6QwRO3BZULYP0z2rmc5TeQPm54GOnFB04aOrZC55po9qDyaWN2jbcjFTjGQcUzrYClFIPpUkUZdsKCT7UCGMNwI9aqwA3NnNZDPnRt50GO5A+ZfyAP4VpPC0bhGHzYyRWVdq9veeZESrA71I7Grg9bGVVXjc9S+GnjxLyKPSdXkAuVG2KVj/rR6f73867zVdKh1CIMcBh9yQdvrXzdfQEKl9b/KrnLhf4Gr0Dwb8UHs44bPWlZ41G37QOT/wId66U4zjyzV0eRWw8oy9pS3Olu7Kezk2TIR6MOhqvzXcWl3put2gktpYponHGDkVnXvhzJJtm2/7LdPzrz62XzWtPVFUsbF+7U0Zy5pBWhc6XdW+d8DHHdeapMjKcMCPrXBKEou0lY7YzjJaMZnr3p3DHjijHSnqvPA5pDYLHnvxSsgxzVi2sbyf/AFUTEnqduB+tbFnoByHu3z/sL/jW9PD1KnwownXhDdmBZ6VLqEhjjGE/icjgV2Vpa2+mWYjjICIOWPf3NR3l7YaNZNLcSxW8EYySxAArxzxp8QbnxHMdJ0IPFaSHa0vRpf8AAV6tChGgu7OGpUniXbZFr4ieKf7dd7axfOm2smGkHSeX0HqF61yOnjEW49TzUWqssKQadbnMUPyZH8THlm/pVqFdqAVFR3Vz1sJTUFZE2aUHnrTAKevFYnckbnhvURaTmOYFom52j0Iww/FSfxArWaEW0zxAhtrEbuzDsfxHNcnC5R1ZeqnIroI7rzLaMj723YSO+On6EflUX6HNiKX2kWAywy5PI9BzTkHzEy+vAqj5u0gj72KjkmbcQCSe9UjjabNIzMvyhlLdue1I0rbypzuBx9Kz1zw2ASDkE9c1ahcE5KgSdsd6oi1i0s8fKvgjkZNQ+UYlBjIKYyc0x1if55AFb0HelS42qcrlRxg+lUiWV5jHLI2wGMY4GehqWLzAmCc++M0KLaZt6blf+4ehoM0i8CItjuOM0xWNBzufbF9WOcAfif8AP061DMqw4JPmS/wr/j6fz/nTFuJCgjjXaD1K/wBP8aesTjAhxzzx1PuT/n8etZmlit5e4h7kBSOQAeB/n/OanjDKCXlKq3XOM/8A1qkaFI9rP8zE8HHU1YW3TguQfrQ2GhlarmLS7gRRBI9mMtwWyRz/AJ/+vXMxq32Y4HCjJ9q6jxDPGNKlRAGLY+f/AIEOg/rWbpNqJtC1KbbzFCgz6ZP/ANakloddCSjBvzMGgUtKoywFUdJLbQPPIFQcmtlTbaWqkgPMeuO1QxsljbbgR5hHHHIrKllaSQlmyxpbmb9/0NDV5Uk1Sdk4UYH5AA/rWfND9sIiQjzcfus9z/d/H+eKY8pYlmPJqhdXWz7hOQeoq4xbYpWjGxZsJ1KvBIBgggo38qoXlmYGZlJKfwnH86uSqNXtHv7U/wCmQc3cQ6sP+eg/r71DaX28iKZC+eBjrW1mtTlUlLYdoevahoV0J7CdkPOVz8rfUdK9E8NfFiYSJBrEabMczJ2/CvLtRWKKQeSSQeSCPu1VLAKDuB9h2rWLe6MalOE9JI+ibD4keGr3AkuVhc9pAVrZi1jQ7sAx3dvJn/aU18uF/Q0nmEdD+VXzye+pzPCU/sto+qM6QecwfkKZJqGi2g3PcW8YHcsor5ca6mP/AC1k/wC+jULyM33mJ+poXoiXh11kz6O1L4i+GtPUj7dHKw/hi+c/pXDa98YppA0ei2W30ln/AKKK8o5PArS0vRbi/cFsxxDq5H8qJS095jjQgtkO1HVdX8R3qi8uJbqVj8qZ+UfQdBW5a6eujWoWI+dqFxiNdozyT0X/ABqaOXTNDiMUW3zT1PVz9T/Sk0nVxaeJrO+miEhhYMkZI+TPQkev8utYOTlolobpKCuyprmmf2Zrw01m3y2kS/aGByPNYbmH4ZA/CnAcAVHcXLX+r32oSnLXVw8h/EnFTLWVVrm0PTw0Wqa5txwoFLigCsTqHKeRWjauQgABI3D/AA/rWcByMVdt+YyBwSeuaiQpK8Wi2wJYEAk/WiUBSp9ufrVpYxGCr4ZmqOa2G/jnuK0SPKciONpHQbdvHHHYVZ/dpGWDgSAZHNUZibdeCcN92q8cm4Z3ENjkZ7VSRD1LkF2C/wC8T5s8e9SO7uyuqkHuD1NVIuTg9OxNNeWQN99go6EdTTC3YvfJN+8C7TnnHGTV/wAwnBLHOP4elZQdQPO80HjAX0qVbyQqOn4c1L1KtZG9anSYMrdyuXyNy46+59v889ahvtcsriWO20yPfuPzYGMD3qhf2d5fsq3kvlRJwI0TB/Pr/ntTbWyFqAtvHgKep6n39/SqujLld7tl4NKWDvgMT8vXp7Uk7yBiZdjleiKen1P+f6UzzCZTg75CAHZR0HoP/wBf1PalkJX5FXDH8f8AJ/zxUWKMvWXX7JLvDGVl5UHhRkf5/wAOlW/BKfadM1qzJBaSy3ovurf/AF6w9eDi1lfftjA5cng/T+8at+AtQFrrVpIzHypT5UnptcY/nitIq1vM2i26corda/d/wxhHikzg1f1exkstRvICp/czMn4Z4/SqKozhmHCqOtQd101cmkd2QM2cetUnm2sCO1Elw+zy92V7VUkYmrjEluw+WY4Yn61lXMxJPNWJ3OOOwrOlbNdNKBw4ip0C3vbixukubSVo5kPDD+R9RWkt1FqX76CLyLgHMiIPkJ9R6fSsYqXYKoyScAV1+gaHOyi1tIHlum+Zwoq6zjGN+pz4WMpTbv7vUwmt7tTK5kXGNzBv4ueg96rzXKu3zwiNiSTtPH5V2upeH9S0+AT31lJFGTgMw4zXNajDbRMr3O4J1wn3m9h/jWdOrzOzRvWoKMXOEtChDJEzfvGbB7irG20I4mcfVayi2WOBgdh6U9JD0NbuBxxrdzSEMLHi4QD3BqaO2sc/vbwAf7KmsrfUlqjXNxHCvVzge9S4vuae0XY3IpdJtgXSOSdl5BPH86ZLquqX+Y7KF0jx9yFCTj6itXwrq9jZCWLVPD1vqETYCh8qyYz0NOv9RN5ei40u0i022zlRC7MWHcAk9Pfio5VuRKo72OTYywyguxD9cZ71pWWnXTkXud0gYMVzljzWx4ijg1S9ur0W8cFzMQ6i3GU6cjHb6jvmuag1C4siUjf5f7p5xWi1WhhKT6mzAQFQDjA5q2jcVlWd0typYsPOH309f9of1rThOV+lcVSLi9T3sPUVSKaJhTqaBTgKxOtCjrVqLhDVZRg5qwG2Qk8f/rqXqKWkWzZWQON3BccZ/u/SqdzcGM4Ulx15PNRQziWMgttbsQajaRiNkgxJnhu2PetkjxxEcuwZic/3c0Pt3E4Gc8HpUDBojuZgKe+2RV3YUD+Hufc0xhHJvysuduePU/8A1qnmyMZJPoT2FRxRjA3Edep7VL95sAcA4FS3qPoRbCd3Xg8k1IWA43fhg1YQYwOCB0p0YCoAEAHbd1pOQWYWF7rYJtyIV3fekKBioPXB5yfzrc8olAZecDlm4/l/L/8AVQjohOF5HGGz+ftRJcnG1j8/oB0/wqpO5klbYj3AZJIU9uOB+X/6/pTW2kAMS7H/AJZjgkep9BSGMDEihHlB+V3BKJ9B/EaERJdy7yQ5yxx98989j/Ie5qSjn/FDM9mdoyQMrjoB/s+3uevauf8AC9wHPkO3zKduc9PQ11/iW1YaZNg5IBJx/wDX6mvMbG6NpqA3fKrHaa6Kceem0iY1fZ1It7PQ9N8Q3QnuEviuRdwhZR6Spw2f0NczdTAoEj4Xqaux3H2m1kibkkiRST/GBz+Y/kKyW4NYJanpRjyqxE1RPUzDNQvWqFIo3RwaqTSBgoChcDnHerd0DVF66qex5uIvdmjoVuHkedhnZwo966TT9QubC4We0laKUfxDv7H1rF0K6aztXZBGd4KMrqCcE9R6HjrW3HDb3VoZ47iJJRuLQM2CFGOfTv8AWueqnKTZ14dxjTSfU1PEfi7UdbtI4b1o1hh5IQYBPqa80v7hrq5aQ9M/KPQVqaxessJhjf7/AN7B7Vkou6JiRjaRz/St6EWveZxYycdKUNEiLoBTgfakDAHkNS+Yn+RXQcV13F3+1AkZWDKdpHII6imFgegJppDN14FFhSn2NvS7rUb+aU+ZJMiIPNbvjoOf5mu28PaBc6nN5Ono0zqoMjr/AA+oHtzVL4Vxw+TfRSRIWvFe38xv+WalRlvwBNeleH7rSfDUFvJZMHl2mC7jDZJZT979P1rnk05W6DTko36nHeK/Ct3a3NtZwwtLcsBhIxktmuF1G1mspZrKcGJlbJRkAIbpyetej+P/ABa+oapDJpySQNbR48xDyMnqTXHqt94g1CK71a42pM+XunTOEXjPTn/GnZR66FRblo1qck2+CQFThlIINdJp0xngSUDhuvsaw9WWOO9miibeiudrYxuHat/w48cGnoJk3FiWA/Ej+lRiLOFzrwEpRrOK2LqdamCjHvTpfJKgxD5jyfao1ODXns95D2XGAOppLwqIUjz8zsMfh/kU+MbmyTWRc3ry6qFhXcsXyg+vrVU4uUvQwxVTkp27mskYj5VwO+Kk3o0J3MFx3z1ogkt50+dSj4wM1XMZSX5irYPTPFa+p5pH5sssgEqHA+4D0qcL3OQSeTThArSbnc7T0qTnzCFTd2U9h/8AXpNroVFNLUUOoIVRk9AM9KsRr0A6989qgWIoSUK7jjJJ4FSx48znn2IrNlIuRQ45yCAMniohz0z+AqTztiEJwT1PWqLvKGOFJ9cAHFTqx6Lc3rfYxzbHAXhpGPC+3oT9Kl8lWyMHH8We5/2j/Qc0lpb+WoeSQMMZUZ2j/gOeg/2j+FStJvGxOQMgbRjA7kD+p59fStDK4jN85jU7j0IC8Aeh9Pp+fpTxE8brn5yT90nn9Bn8P5UWkMjsB5YVV6Mo6/5//VVlZ1RZI7VVMi/LvYDYjen+03sPxosQ5FHXVS3sikgBuJo8pB7ep9B7V4nqcbJdPvUKc9Bmvco7EmR5rhnuJpAcluuf8/57V5d4101oL15Ozc9MYrpw8lGVjCtFyh6EfhzUFlUQzMdy8E+3r+FaV/avbTskgGeoK9GB6Ee1cTBNJbzLJGcFTXf6Le2+s2KWs7qkg/1MjfwH+63+yex7GivS5XzLY7cFiuePJLdGWRULir95azWszwzoUkQ4ZWHSqjrWMWdzRQuFyDWdIta8qdaz548GumnI4cRC+oWM+zCbtrA/KT3rTvbwXCs0kKpN8o3INowB6ep9awnWkMsqqF3kgdAa0dO7uc0azgrMl2xz3INxIUiHUgZP4CiSTdgKNiLwq+g9/U0ljdC1uRM0Ec+AcLJnAPY8EcikiuikU6FFYygDceq8g8flV2Odyu7kZUUhAo3UbhTENIpKU9aSqJkegfDS/sk07WLK8G2WSMmCUDONy4K/iQv5Vu+GtJuLhJFizvRCz7uy+ua8y0DUotM1SKe5jaW2JCzxocMyZ5wfX0r2bUJ7K50tdS0W8KxCPaDDyZUHTcvZh0P0rnqpp36BCzFudF07SLW6/tmdHeTczKh67Fwq/izf+O15/wCI/E7XUdvZ2YXyreFYEK/dUD+uSTWfqN9e6rM65lfccbQxPH0qtptlHPcoLky+WhywiQHAB5JzgD6k4osnv06GsU4/Dp5lK2tvtF0OSy55c9/U/Suht9rDMalY+iA9Qvas/wDcyXEkFgS0O795L2Izwg6ZHqe/066sa4xjpWNeXQ9LAUklzIsRcHmpMc02IA8t90frUrMkcbTSkKqjPNcTPUvYh1CcWdk7AjzG+VF9TWLpQ2XCvKMg9R3NVr6+e9ut+MIOI19BWvpscbDO7c5GCK7Iw9nDXdnjVq3tqumyLaAySEr/APWq1EFjQ7l5z/nNAeC2Hzt8+MhajZxIfutg8ntk1i22WkkPebcw2cntgdBU8cjqMZCpjv1NRRjylLEBQemeT+VCt5hBH3fepYx7N5p+WrMO0HJGSOgzzVYTJEMRjcxFNkl8sZlJBbkCpaGiy8gHzEbVz1P9KkjExQeWuxewIJNVraPpPckkj7qn/PWrkdvcXK+Yht1XoBJcKp/I0+UhzNVLsXEmYSzKzYGOp9//ANf6CrSWzScDcowCcdPz/r09M1LaLbQwlmdEUdSRkfTj+VTQzLO5jtgxXO55TwD+J6/jwK0OdshuFdIsb/LiOdxVsMfb1A9+p9ulLbQOdp+WNFGOVxgegHYfr/KpPKsluUUzRPMQWABzj3z0/GrMsiLtTG9/7o5/yKNRXJI4447Ys+NpH93r+H9K5Pxxpp1PTmkVGMkA4OOMenua6QbydhO4nrk8/QDoBUktj59kVuCCwUjAX+XvQnZ6BbufO1zC0MhVhgg80WV3JZzCSM8dx610vi/Qn0+4LKmI25UA5x7H3rk2UqcGvRhJVInJNSpTvE9M0rUrPX7JIL19kkYxHdHkx+iv6r6HtWfqenXFhOYrmPa3UEchh6g9xXF2N9c6fP51rKyMQVODwwPUGu40LxRb3VstjqUQmg7RFsNGfVG7fTpXJVoODvHY9TDYxT917/1t/kZTp7VWmiBFdVfaAXhe70mT7barywUYki/316/iOK5+RMZzWcZHY0pLQxZoSpqq61r3O1VJNZMp5OK7KcrnmYiCiRGkpaNtbHENop200bTRcTTG0EcU7aacseaVwUWyLbV7TtWvtOVo7ab90xy0TjchPrg9D7jmmrbbhxwaelk7EDApOceposPPoXIvEt5E0jxQ2yPIgRisZ6evXr71Er3+pgRzTMtvnOwfKufoOpqe000Fhkbj6AVqxQCPjAyOw7VzTrRXwo7qGCb1mxtpbpBGERcAVeiUtwBmmwxFiB+pq1KYbS3eWY/u14LnjJ9BXFKTbPWSjBeQ4xiGPfMdoAzz6etcpres/bp/ItziBT1/vUzxD4iuNWmI8x/L4GWPLY6VmWkW5gfeuyjh+X357nj4vHOo/Z09upoWcRmcdeK6a3VbW0B2HzJOIx6mszTIeVYpx2U9z6/StNfMkZ9oLEjlug/D0FTVfM7E0o8quWoIiEKKQzEktKRnJ6ce1TIjDlcgY/nU+ntHc6Zk2zRBXPkuDnzQByT+P86lURtbNN3Q4b2rCWjNYO5TMeeWPbP0pYovMhYk+WqtwpPUetN3PO5Jwq9u+fp7e9SSEADeD04ApPQpajZdsagIPnJ6D+ZqOKElzJM2WHIz0qWOMv0PJ7VYMQVecHHOfU0r2G1fQjMquRxwOee9SrEWyQCefU1CUDPnBx/npTS6qSJMswPPPSkxpI60YlZIA+3cemc7R6n09gKuR2iFGiLEIuNyg4A9AcfyqOwjES+bIu1cfKoHLH/PerS3CNGpO0IoyFBwAPUn09+9aI5Gyq4jtnYqDhyFeTGWOO3/ANbgCrNiROjGGMghu4/n606GGO6VkVfl4GduM/Qdq07O2W0j2I2fmyzep9BRZsTaS8ypLA8QASNpJ2Odgbao92bsPYZNJKJ0G4OrSnjlTt+g9PrWjOyquenr/wDXNUbos4ysmAMFhnqKbViYu5z/AIj0YajasrZJA3BR2PtXjOt6dLY3TxyKQQeOOor6JtFRoxLLnqV2t0BzXJ+NvCcmqI0ypHG+7hhnkeuMVdKbg79AmlNcp4cOOvWkyQcgkEdDWjq2lzafcNFKhDD9fes4qcV6EZKSujilFxdmbejeJrvT5kfzJAU+7IjYZfxrtjq2ia1DG2oBGnkX5rmyXZIh/wBtDhW+oxXluKVHaNtyMVPqDWU6EZbHTSxk46S1/M63xDpYtiHsruK9tyM74gQyezKeQf8AOa5lgSeatW+szJgTqJQO/Rh+NXJb3TruUsqNED/DIdxH/AuM1MVKGjRtKcKz0lr5mUqZqxHbFhk1vaZbaHIT9ta7XP3Xt3RgPqrYz+daX9i6XJ/x661GPRbmBkP5jcKmVWxtTw66nJfZG7UhtW9K64+G5T/qbzT5QehW6Ufo2KQ+GNSHIiiYeqzof61n7dm31en3OSFq3pU8NkxOTXTL4a1D+KONR7zJ/jVhNAkQAy3NnGPedT+gzUyrsuGHprqc9HaDvVqOAL0FbBsdOg5uNSU46iGMn9Tiq0+qaLaD92jyn1lfj8l/xrLmlLY3/dw3IooZJDhAzH0Aqx9mESbpmVQOozWPe+LSQUtYwinjCjA/IVz95qd1dn97K2OwBrSOGqS30MKuY0afw6s6W/1+2tQUtx5j+3SuZvtQub58zyEgdFzwKq/zpVXJrsp0IU9tzyK+Mq13Zuy7Do0LEVt6dZch5PlRRyfX2qtZwpEvnTLnHRa1bITXzhUQqmM46Z+pqak30Ko00tXuXLCM3E6xRMEQ/Lu6DHU81Zt/Lvd6rIYrGBdzMThpv9kVbWwjKJGV4A6dOPf0FaOmWKyOF+zmSOI7mVRy2PXsBXI5I6XcksZwbYS+WUdVAj6AKvp75qsjKZCJCVhkPz+h9PwrQvGUyuAuBngDtWdJxgvggcYrG5rFD33xEo6gEHgDvUYt3eTfn5mHPPSlVllYONzSKACx6Y7AVcg2BBhSCSe/86Nir9CAgRYA9OT61CQ8nA6k9KsXe5E8yUZI6VXZjCnX5nxkeo9B7evrUleRNGpjZc/6xh8vPQep/pVZyitiOVgp5yB196X94Syt88shwxH/AKCP6+gqncXk8MzR25QqOrbc5PenFNsHsd2qtKCXbGfvFv6+v0q9b2yOwDqCPvhO5x0LUUVqjjb0NWDC52xrGn94dXPf8KkV1Lbic54BJ60UVRmxzorAK7HcRwoHQetRsYPMkjZ/m+6Bn2ooptCRIbBZ8M8jKMc4OKeMXLTQTr+6QDBzkn3oop2SJu2cN4z8MLcLIyw7t43I4GW/GvJNR0e4tHbzI2Cg/eIwDRRTpycZcqNnFThdmW6EcVGRiiiu5M4WtQxShSegoop3FYUFl5BI+lSpeXCfdlaiilZPcpTlHZky6rdr/wAtM1Kms3mfvCiipdOHY0WJrL7TCTVr0Hl+agfVLx+szfnRRQqcOwpYiq/tMrvcTP8AekY/jUeSepooq0kjFyb3YlLRRTEhyoSa0bS02gPJwO2aKKyqSZ00Ip6mhbWcl/cJBaoTk8D19z6V1ltax28SQxnfhPnbpk+lFFcdVu9jrprqSNcWyTfZ93zbSzY7HHAz71bsJpbSEEsY8Ahu24HtRRWMlZI0WsncZvaT534Yn5UA6D396RkCoSxycdAOoooqOprbQdagoQ7oQOhX0BqUYQnzG+Qc/T/9dFFLcRWvpPOYyMMRLwAerH0/xqGOPYN7/wCv9euz0wPXsBRRVbAtSO4YQDaHxJ/Fg/dHp9fU1WEYIzIxUnoABwKKKqOw2f/Z', 'chelsea': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAkGBwgHBgkIBwgKCgkLDRYPDQwMDRsUFRAWIB0iIiAdHx8kKDQsJCYxJx8fLT0tMTU3Ojo6Iys/RD84QzQ5Ojf/2wBDAQoKCg0MDRoPDxo3JR8lNzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzf/wAARCADIASwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwCaS2KniheODWzPbg9qoywYrDlOm5XY4HrVSO4VpJI84/GrMp8sHNYYkCXUkrHAas6miNIbkk8hV2VzketVQ/zqTzkUy6uUD8k4pI90p4AHpWSOmOqLUdtumMsTYzyp9CK6LT5EuIFZgNw6ms7So0MeG691rR/dwoSuOByBWl9DOW494hk5A69agmiiwNw4zkD0PtUZvAQWPbsO49apyyySTbIyGjb5gf7p9KnmFYfc3YUjk4U9u4plxdyRfcGckDntxSfZl+bK5I4wD0HWoftcTq0y5MaygHI6cY/nUjAxSOuxjgFTkenNZ7CVRLaRkL8pLE9Rmprq92/u41Ik4yT+ZFSD5757l/8AVMA3HfjikBRtLd42Nu4IWJdzknmpYJxPI+Y8BF+8Rx7/AFqyA0+oXCEfIXAPqSBwP1pYglu0hlAKA7FTHQDqfz4piI45YTHEGUAZ3jI4A7A+taVm6qkxijcgbmzJ3Oev0qlDcWzzM/lZycZbgKB6VfZ1L7C2DJEWYA8AcU0Sc54rLR2aSyY3swOB/StDSdQE+nAFwzt1GOPYU3X7S4urfOwBm6HPKr/jTfD+l/ZrNWlOJ9w2rnoO9O65RWfMUNW0zbfS3OSVQA4Wo5F/0cMhyrjLD+61aHiZ5ENvDyGdmLDGPlHr7VSiVbdvOjyYT82wjjJ5ou7DDTp7h7KWOQ7ZI88j1qfQ9ca5DW9yD5nQGo5WFtdp5aZjnQP9CazEsmg1gmUMiyEsrKeKGlZi10Ow3oJsBQFIyGFOZg/zB8gDms1PMhhWNH3EAgMaSO5clVGMAc+5rGxqbAddoxwxp6XDLwKiiVZdmPvAdqsW6kybdnHrW0GSSC4lUDgHPJFBeOT7wKGpCSG5VTSSSiMYEQL/AMq2QiORRGMthvTHaq7lmPTNO3Z5CEGlMgjQvIAABTbsWlbVlaYx28TTScYHArh9SZ7y5aRicZ4Fb2qXLXLlVPyCs0wj0qbmU3zGQbY0htjWuYR6UhhHpRcjlMj7NR9mPpWt5I9KTyRRcOU9f3jGDVeUA5p0gK1EMng1qyDK1MhYmrl7vzHYCMfL3NdNr8bJAWAyO4rlkmLfc6Vz1Xqa00MSxeTJZsk/pWpYW4jYJMgOOQ2aks40ePOSGPUjtWhDbEKVmUMD0NYJm1yykUQXzFYD2NZd47pIWL5VuPqKW7hkhIWPzCp/ECmi1ErCRXIx95GPH1rS5JNbwbQjwSblH3kbqBRdRm3lRYV4zkYPUU+cDyf3ZClR83rj29ay5Y5gVkWUOqHOQelTcYtzfZvnCqwUJwM4ORTBteBx91ZSSQexyP6/zqVIjIyyOFJBPbmrnkrHBulRWB646H3x+VK4EKWio2505Lbtvfpj/Gq8ssMJa3UHaACreuAP6UgvGDuDyVJKkHp3xUOzdItxEpUEksrcgUXCw+e9MUySxj5OXJH8R7/lzUen3AuLp0lBJY5BHQAf/r/SoHDhDlTsJ4z9eRWrbtbImwkA4Bz79/6U7oViGaEStPCnypGUXA4zzxT7e1YHzWlIV4WiJ7gLknFWpVxFK8YyfkY8dcdR+QpZQYoUckbRCw2E9yAataEso6xd3C6SWijC8cD0X/GqehXc0d1GXXJnjLMxGStX7W6hu/MidT8wLAt0VfSprqzMcYkT70gKpj6A/wCfrU3HYlurdNQjkAy0kQ+U9znmsW+Tyo1t0G0vwZCOOB0/lVzw5fFpZUlbawkIk55pdUljt9WeGcgxBQ6rjpngmgDD8yaZQ+3eVOzIPQAZBxWqIGvdORCmJNi8n3NQ6e0cCXE20lVDFT6A1rxShNPNyRlGUYI4+lNiMoqY40hZj8uSSPakQKH3JwpxgVUtNQV9QlGC4YnIxwMVpyxwmUqvCAKOPXHSpaGmWtNnKcMeWrbglURAEfM3P4Vg2sBa6QHjeM/QVqWyM0rSuMRjoP5ChaDZZmXb8wPznpntVB22OCT+NXJnBG6Q9e1ZlwxfPGFHStOey0KgrsubhtLVlX1w8ylR92rluDLCU5qnNHs+Q9qItyYTskZpjppj9qumOmmOtDG5TMdNMftV0x0nl1IFLy6Ty6u+XSeXTA9PkgBPSojBgdKvkVG4GK2MjnNewLZhjPFciIFLA45Ndb4iidkwp71kwWodCG4A7mueq9TamVbUMXIKEKO9aoJ8n51zzz71A8qRHYgGfapRL5ynYRjuBXP1NCtchVIaBmDZ+4TSuxYYkXbuH3lHeplCsAinnuAM1Xlujb58+ORVPTPA/SrRJUQSQSMImEgHt0/A/wBKcAjpvePy37BeM0iObl3WKFAme7EH61ZwbcKrjG7uykk0mUhsKoU/eKyknqOAahuFePozYb+Ej/IrRVEQZCs4bnJ4x9BVdbdp52jWPYOo3P8AN9eKkZiMQCEkz/sv0z7GrAYqhykiBRncnzBvrWpd6bJHbMGIbv8AcJwfWorKMugDRuko6Oq4B/CjlHcxWka4AKsHQNlk7/TFXbL960qBRvK4I7ZHIP1/wo1WIxzLM1qRIOSy8Bsd+KdpqQzEyRlhnBDZ5z/WnazFe6NMSEeU6hTDKMAf3Wwf/r1S1RmFsmXCsc8H3GKv26bYyAwIJ8xfTPSs/XiDHGV6tkfQ1UnZCSuzMsVYxk/xyEA8du9ddCizxROeNhbaO5Nc3YL9okTYMDJJB/KustIhGQSflC4GPpzUxdxyVjiIrRbXVrp5mLK8oZfQGrWrwwXEUGp3LEpnBA/iJ6VD4piddTjCEqgQE+nJxV+9hWTRIVVfkGxVx0+8DmtLkWM+6gMemSxhSqlQBjuc4x9KvX7unhyVIgWbaMAj0qKS6ivLKYHHBO0EY6dKTxPHNFpESwMV+Xa34Y/xpJ6g1oM8LaeGiSV49zNyzdcc1cFqEu7lF5QuCo+hq54bhEenLz0XBPvVO5WWC/jUyELjc+T25oTuDVjSig2JIV4kdQF+lOeUo8cGMBRyfU1FZzmSbzXyFPC+1O1CNgF2MN56L6U2IlukYKGAznpWed7Phq07WVZkCnqBUDJtcswwoppXLUrEsEaxQliMcVkTOGlJ9alv9Qz+7jPFZcs+COa3irIzleTLuAaQpUMUmRVlOaGQ1YjKUmypytJilYLkOyk2VPto20WC56MTUUr4BzSlqilO4EVoyTndZnJlWPeBk9KoSzqoVGwfb1qfW0XzQxGW7Vlid3OI0B29c1yVfiN6ew+4JXJVOO5BqGJ5YBlJOGPKmnu8ZQliMjnioWmRwWDfKO44rOxdy1FeRx5L7I2Iz71Nb6rYyMEa4VZAergEH8a4+7BnnOGLBf7owadFYkldzEqT8ueP/wBVVZIVmzsZ0hZ90UUJkbnzEOacu5iFZ1dR2yBtrBi0+dNrW0zITxlT19jWrBbXMCHciyHvkDrWbZSiaEVvEw7t67WqxHCkaHy1UE+39QKpC6nhhO+DaR0wa5jW/FuowOYIUSMZwAoySaqGuwpK252dwUeIpN5rZGD5ZOP1qkz2S7VDtGRxgg//AFxXF241u7Pn3N7Iue2aW5t78Lua6uCB33mqv0IudkJFiTZO++In5ZNv3fbimi0jt/30ahBncQvf3rz86zd2U+1buQS+jc5q9YeK7mCQC6iWSE/eZOo98U+V9hKce53cUYVHI+bPI46E9awte2sEVmK89u3+c1sade295bpJAwKkdqztZtxNnjms5s1iiTQbQE7yPvHvXTBNkQAGTWNpMbwod/y44FXmmJJGT0/XsKmA5GD4ngkuXV04YYHsBmptQDLp4iUA7VG0Djk1qyRb3Rc5Bxu9hWdqNvulLqCA2M5Pbp/Sr2J3MuKzWKZ4sDZJMCuBzgjmtjXLPdE6J842cJ79asW1uGjg3YJVgc1oXEaSQM2fnGeRUptsp2SMnw0SsDwlSMEden0qtr0TLcKx6McEj+Va1nD5LbcjG7cT3NReIIs2W5QC65K/U1XQnqUdPd5JNqJhMD5vQVpXKxSMBGuTj7xrK0TzAAhBIHrWzNIgQF9qsfXr+AFVB3RMlZmfCqpITkA98enpU18BJDtQ9fSo12Fs4+UcEnj9KlUK5xuHtThuKWxjtY88H9Kgl08ls5/SukFuCeMYpxtQR0rr3MlKxziWpQdqkQbTg1rT2wUHisqZvLakxOVx/FLiqvnDNTRvmkIlxSYqReRRigDtQciq91Ksannmq11qKQqRuFc9fauJSVVuauTSBIbqtyJJxk9+tVDFtbemMGoLp28vfjJqtJqcZttuCZBwMcYrkm7u5vFWRau7i3t8l22sRyB3rFutTjnTyIlwScAVGY576b7p2dwTWnpWkOspd1VyDwfSpukVZsk0nTFEQOeT1I4q5f2RWJSgGQc4+laCWsikFRgdx0zVi4hVo8DGR29KTV1djTszNgZJIuOGUZKg8keorctSGgRmYMx9O4rnpd1tdJuUAc4I6VpWc6KEEbAjPIP8JrKL6FyNK6hUxYAzxxmvN0gW98TSI/8Aq4mPWvTUIkjOBzjt0NeeBPsHi2ZJVx5jZH41otE7Gcuh0LQRJGAgHArO1zbZaY9wWYFR2PvWjcbcZUEVyvjW9laxSGM4iLYb1NGHSctTCrLQyL+WC8We4t4g8HmEK0gxIo9yKbZQmSISDkdDWXaTGCKUcFZF24z0PrXT6NBttEVsZb5sV21n7phDcitbyXQ7lJkJ+zucSJ2HvXdWU8V7FHKpDK3f2rjvEUKjTj2x+tWvAN1uje3cklTkfSuWPvRudkXZ8p24XAOOnaopBIflTIwwYmrKhtnT3qvPuiiZ8nOCcUmrMu9yK0meSSRwMKeOPXHJqckbQgYtubr6ADpVOAtjyx8pBHDHkZ5OaWNnVtzNyZCrccADnj9BQ9ieppKpxwOgqaBVRJOpDnPPrRGu6FS/DFc1GzEREYzjJH1rOCtIuT0IftCmRlA4puoyM9k2GCsR19KgKecSUyoLZpuqXdvZWpkupFjQeverabJTSE0mIRRjLdTmtEvB6Bm/vV5/J4qkuZfJsVMcY/iYckUk8mo3Cb47yUMBkYNNRcVqQ5qT0PQRH5gOV4PvUT2/lMD1FcX4f1jVHkMc1wXCnBVq7e1uhOmGPzDqKeidh2bVyaH61ZwNtZzTbJMbql+0/LW8WYNajLzAU1zGoygPg1tXtxwea5bUZt0uK0ENM538Vdt5jWfCAeTV+HAFJjRoxycVJvFVFcAU7zBQFjKvNWkuJdobAqe3hJQM3JNY0Nt5kwzKvXtzXTW0cZjVSWOPQVg22aoe9sDbn1x0rMt9KiuJe+4GuhjjUJ84IB9TUNv5EMxXeoOfc1DRaY6001IcBwCfarotgp/dDb68UpAYgx8+4qxFEWIJJzS5R3uEcIVSr4Oe1UborE2GJC9jWsygDDfnis3UELD5SPxptaAtzPvrhFAzhxjIIGfzqtFOpQPAORzj/P8AKpZo/NhMcijcOQV61T2FJFwp2Z+YjnNY2LNjSdRQsY+FY87SeM+1Z/jLTWeOPUbVd0sByQOpFVPs6RzDa5XJypYcH2NaunarGXe2vAwXoWboKq4mrlPTrs6naI6MNwHSsrxFpzT2ckfAfqOO4rUvtFmsLh73Ripjb5mgzwfce9QRa3Y3KeVeHypB1WXgg1HvRd4mMoX3OG0/R5Gk3XA4B4Xrmuw0uwOATwo7mrnm6RBhzLGc9w2QKz9V8S2yKYNOALHgyEYA+laSqTq6JGcYKJDrhju51slYBFGXYdhWf4SlFvq4VT8rHb9apT3SRwsiOJJZfvuD932pmjnF6jdMMOa6FBQhylRd53PYogGiznP+FQ32Y0BUbtxwB61JYyB7VHXBHX61LeAvF05rOexstzGlUhSy/e7s3Tiolw4AZ2KBsbgOozyfxq5KimMKx4PXd3quqZZiZBtY5wo+6P8A69QmDRrgkrkZIHrUM8hRPl5PHSn27L5YbdnPbNVLm5VWIUbsdxUrcp7CFxbWj3MowBk8153r+ptc6hE90N0bHATPCiu48Tsw02CMHhzk+/FeeeIYisUcu3dtPNb0/iSOWtJ2GbFg1ZVj+6SMfQ119nCCoriNEDzXQdlLBe9ekaTABGZHGFAySarEE0n1MKwtD/blwq5GMGuhRL23JZFBqp4cQXmrXVyB8pPB9q64xAIM4rBx1OtPQybeGdwDKoJPWrQtjt6YrSto1JxxV0W6GuuK0OZ7nKXdizA8VhXekyF8gGvRms1NQSacp7VVgR5x9jlj7U9dyDkV3EulKf4aoXGjgjhalplpo5Rpsd6b5/vWvc6M3OBVM6RID0NTZj0MrTxEJAYlLH1atyCYs2DwPasHSraVfm7VtW45O/j3rFjRpRt+72MM+hrKvXeOXcByKto+1vlbIp15Es0ROOTUblrQk0i+WY7D8rexroEGF+8PavPY0eCcyJ95T0xXQ6bqjNgTPtI7GjmtoNx6nRdsbwKr3CB1OQD7ioTeFsFSD9KrS3YyTv2ke9U5JCSZVlcQy7ZVPP3TioXWMKxBJ3gbWXqDRfzrNCVLrnsRzzVK0uwPknJX/ax+tQ7dCrMsBLiNXVyFCnGR/nioljUMspw6AdVXjn1FWXmikRRG7YbglDnP4dvpSqxhkVEEbpJxtxgmkBLb3b24byTuj7xnj8vWrFzpumaxEBLCgfHToRWfIMS/NHKrdMqQR9CD1p4SZCjK+4578E/hmpemwzOufAcZYmGRlX0zmoT4H8o/NJkDqTXTW+qsuI2L7v7rDFTTaj5kJGDk9BTVWWzYuSPY4jVNChsYv3Zycd6yEUQPGxAG4811eoxvM+Xz+IrFv4N5G35VUccdTVwq9GOUFujuPD0vmW0fO4gfLitW6cYCbvfmuO0HU/IhKMcMO4roYblpdxdeQPxqOfoNxtqK+WIXbnA61AwJ4HAAxgdqejB3Krjg1LGq9GIwPUVPOFhLYqqZwenp1qsgzKG3DntirkUXmbhGuT2PSqfMMhQADHqKUZBJEPihl22oYgDkY/CsG80z7TbkDBVhyKs+JXOpahBbRyY2rnI7Gs14tesiVRBMo6GtW23ozCUPItaLoQgxlQqD1p2u6yuDpenfO7/K7L2FUxaeItR+RiYYz1ArotA8Jw2X7+Zt0vXcafnJ3YowZpeGdMSxsUDfePJNX7y4RBgEDHpVS71FLVCiHc3SsSSeS4LOzkAngetCZozettQ546VfTUQBy1cvG+1cZoeZscNXSpWRg0dWNTX1qePUEbuK4OS6lB4anw6jIp5NNTCx6AtxG3cUrNGw7VxcerMo5NPOvY4JquZCszqJY427Cq5t489BWCmuhv4qmGsLjrT5kBjtCsUQEfBqS1V2QhgCKGVjkVLYkNlejCuSxsV/L2yEYx3Bq4gZ0IbFLLGynNOj3YHHSptZlXujFu1CTfMCDn86kFpHcxhlz/WrmoQiZdw4f1qtZOYzh1B96TWpaehXZbqzxt3svfmq0t5Jkg9fYHNdQojdOgPtUMthG/OwD6UOAKRzG5pudxB9SOKt2sG7Cy4I9UPStRrDByrEe1QPA8R6BvwpWsVe5BHp+1i8E7B++Bk4qyd3lbHnPI4ONpH4jpUQtWP72Hcp6lc8Gorkz5xIr49c/dotYT1LJnQxgSuzugwrIT8wqWLeHUPI6Rtkh94I+hFU7ASEsG+RsfLJjg/WpvKnZnwQ8i/NjZ1/oaCS1y52yyI2TkYyD+WRRLDIrqqzZY9cA4HtTbc/vF82A4PUDkH8R1q5i2j+ZPkz0AJIH9KlodzOuomAIJBcnk7ax7yF4yxc9iQAa6G5LBSMB93YjHFZFzh1Tfl2IwVFStCrmdpcRWUuAXIP5V0VuSFbdktnOBmsCW8WwlClf3kgGAtXrK6LsVJO8n8RUyvuWrWsb8RyP3mRk8YPWrYdQR2U8Z/Csu3nZOJAAfY5q5FKrHr+fes3IOU1bJMEgdB+tZ2swGOUOgJ3dSK1bNl8j5M+pqnqall3Dg5zmrWiJe5wshePU0l5BU4rs4LmDyQzDoOc1yd8gju8kdGGec1rrJI8Hy4wfUVepLNT+0kDYjTA9aq39xK64R8qe2cVRjLDcWlwi/wBck/jRJhgcAqwxy1WrkMjwrZDLhwODnNIiuF+Y5xUhZd+3aWb+8O1SY45raEepEn0KrORSbyadMtRAZNaWIFYbhUDAg1aAproDTSEytkngUq2jyc1etLMyMCRW7b6eNvSny3Fc5dbORT607yXHGDXWf2cPSmvpwz92nyhcwmLMpOfwqOwdkuDnkGq5kaNyWbj2qW1eMS53EZ7Vzrc26HQFA6jNQPGUPy5H0qW3c7eOR71YZGZeF5rVq6ITMyZcr938aybqCYNvQH8BXQyROQeDmqstvcNkYOPrWTVzROxlW93cRAZKqO+6tKC6M64yM+vas+50yVG8xYi7egohW94H2Z19tvFJXWjK0eqNJiwfJ5FTBI3A4qOFJio3wsD7g1MihPvDH4VSQrjlt1AwoyKiktgTwKsLz0xT8etFguZv2PeSAxB7ZoigeMGMo+D1IJ/xrTUqpyRkHrTZXhPIkZj7nispaDTMuRDKQhUKCcZwTUhUBSpYnHUD5sj/PrUh2qjBWGSeRtwMVAWEQZSE8vuSP61nzFWGmIfe5TJ7jP4VnGJhIMyFySee4/LpWpBLbzKdkwwDgBQMD6VI8KRhfLIHYkDpUykVFHPXUMCSmSRBuAyD6is4Opk81Bhu1bl8PnO8YXrWLckB88Dnv6Uou5pY2bS5LKN/pyQOa1YX4DD065rnrCUnYDke4rZgIGACcqemKzlELnQaaSFZeM1T1h2XIY5XGRipNMlCud5BzxnGKj1x1wVXgjv7Vpb3Lmd/eOSlnaW7ZFGFHC+9bFq5ki2kYxwcetY0cJ+1S5YYJ4A/wA8VfhJTC72O/qB0xWiJkiSRyzlCDnrzUQlE5MWT14qXyXOVCDLHOcVbgsfJw78v61rFGbdieys1CAKM+9TS2Jx0qzZOBwa01jWRa7YJWOaTdzlZbJmOAKlh0045FdKLNc9KlW2UDpT5UFzlpbLaOlUzAd+MV1txbAg8VlTQBWzQ0rCuLp1uABkVtRIAKz7PAArQRsCpRRLtApCoz0pA2admmI86Hl7yPmkx68U9p4lICooPrjP86ryuik+YQp7YqFuG3biVNca3OnodJYvOAGDlgf4TWpGxI5BBrG0d1KfJIcjs1bMTseuDW62MhJYww6VRuBsFau0kcCoZbYtzjH14qZLsUmYzbW4IP1qhcQupyrfrW7JbHpvQfrUTWkQGXck/wCytZ2NE7GLBfmA7GZ8+xratLt5FBEj/iazrq2tVO4xTOe3zBaZbajFC21rcr6Zkz/IUJ2G1fY6OOdyOufqKd5rH7yR49wKpQXiuAREmPck/wBatxyF2G1UH/Aat7GZIyLKvCL+GapXELwAkbsdlxwPxxWlJcpCvLgfjioRfxyAr8pHqQa552ZpG5hw3jPKymPO3tnPP0NVzq9u0z20kLROOhfjNdCqWpLFcFj1K9KoXGm28k/m/KMdDxWVrGlzLttKgE/2mN9j46BwM1o+Qwb92Gz6k8Vl3Wlz/a0ltrhl5+bB7flW1AzI6RyKOn3iMg0pajWhn6lA0NqzlQcdcHiuSnnjmzIpzg42jsa9JktVu42VyeeMAcVwWv8AhmfTpXntSWjbqpHWtKUNSZzdivBfohALDJNa9pqkbrhWI/nXMwTWcB/ev8567u1Xw1u6Ax4x1yK0lGK3QQjKXU6+0ukMW5MdOntTL6dWDYcDI4PUVzMdwYgoB+QdqgnllmP+ualaLVrg6c1qasMbG/JXH4d6u3myIBwRgnt2NQaHZyGAyzE8Dj3FF1axZkaSY7GPTd0rNK0gbujasoVEaPkHIp87gcZpmnqjQADAA9TTnt9zfKU/OuiOxzy3GpMY+au2upLkAkVQuLaTYcDP0NZ0NvOk2cEVrCTRElc7mC6VwO9T789K5y0maMANWnDc571upXIsW5T8prDvpQHwK07iXEZ5rnp5N9xSkCNaz+6Kths1Rtm+UAVfhGetAEqVIBSKKdigDzO7hDgFBk98UeRciEAbAP8AbPP5dasSSF12ISAP7owKqFZo8lcD8a4tjqLFo+2Vd5lJ/wBhdo/WuptGIQEKPx5rk7S6mc43DP0rpdOyyDfg/Q1tBmckaiyMeMmgjPUU6JVx0qTaO1WIqvGDUDRDHSr5jP0+tNMY6kZHqeBUtDTMma2DdqzbvR2kOQAnoXOK6R3VR8v/AI7x+vWqcxkYHadmeu3g/n1qHEpSZlW9stiM3Vxx6fdH68/pWrZ3lmw/dZb1PT+dUJdM3fNIRGp7t1P0HU06OCCy5Rdrf3nGW/Beg/Gkkxtpmw+xx8iqhPQhcmqs1mpB84kf73LflUdvcTSArApU92Jy359qikjkQE+aWyeT2P09aynHUcWQy20h/wCPcEIOADyapql5C5yQ0hOBlsBfc1akuLwZ8sCNT7cmoVmmSXEkW5+pyeRWVjVMRZrmK5VNoYHr2/GtO1uyWEZhYMf4sf41VivI2lwYTvPv/M1fiuYC/IAf2NXGLFKSNBJAF5JHrk1HdxR3EZVmDAjkYqAzKvQgZ7lqU3CEDO7PuetdEUYNnnvi/RRbN5yKNjN2FZFhbvaq25wUPIFeg+I7db3T5V3DcFJXnvXmYvmwsTg7gfmFXKLkrIqnOEZc0tzTMzMPYnAArotB0Qyss1yMqRkKareHdJklkS7u02KP9XEev1NdeGWNMLhfSoVNIupWcnpsJdIixBIsLgYrNt9OjRV3sHIbJBp+oCa4wqNtFV7WBjJiVjjpnPWsZWuStjaWK3EZ2qMnvimABegUilSLZEBncvqOoqGbKrlTke1bxWhi3qOkeM8MCPcGnQoByr5HvVAT7j8351Kkh/h5oAtTyFB0FQwXyhucipEikkH9KhuLML82MGriSy3Pdho8A5rOj+aYtVOV5EbHUe1aVhGXAJFW2I0LRTgVqQjiq9tDgCrqgAU0Aopc0lOxTEefLJO6H92MewqFYMZMny+xp8SmUZ80D0ANT+Ur/K6k++a4WdKIbaA7+FRlz2rotPVQowpFZNukMLYjG4/pW1aYKgluP0FaQJkaCAAVIM47AVHG64wvPvT+takATjoPxNROGJqbHftSFsfdH40DK5i7ucCmHCjKDaP7zcn8KmYjOepqB8ucDJJpDK0rEZKZBPVzyxqt9l6STHan6t9K0CqJ1AZh+Q/xqFkM7lnPA6k9qQEK7pMKiiOIchR/MnvTGYu+EOcdz0HvU0h3fJGML/P3NM25Hlx9O59aTQyu25dzKc46uf6VUxIp38c9vWr74bCr9xf196hkTcSankQ1Iqm4O/G0H/PaneZG65Kk/jimyQZyB1NV2gK8Bj9KrlHcsl/lO5iPTaaQzFlChjjqeeT+NU3SQrjJxUYSROhIBNNIViW9u2WFwDxjbjrWFpWmo1008ydTn6Vqtas/Un1qzbQspIAqkyWi5FIVQDd0FI9w+cBuvWhbRyPSj7Ed2SelQ2CIsu2fnI9Ks28Mqj523d6WO2UGrEZAG3PIrNR11G5aCrKV71HK2/8A1R2yenZqbMjOcp+Iq1a2LOBuFbLQzZSitzOfmXa/6GtS0089xV+3tVUAkZIq4qADIoEV47dUHSq17ECpwK0WPGDVG7YBTmrQmc4bc+f+NbljCFUcVjXF2kc2Ca0bTUY9g5FCA2kAAp2cVmHUP7tRveyN90GrEbIIpdw9ax45pm9al/ennBoA88s7ZoyPODZ/u5/ma2oUbgFwMdB2FFFcTOlFxIwuGkIx2A6mr8EqEABSAO1FFOLsyWaETj+EVOrY68miitkSKWJpvB+lFFMBjDPTgetRseCE4Hc9zRRQBHs3H0UdTTJOQFUYUfqaKKQyB+BgfjTW+RNo6nrRRSGRkYT600DIoooAidTzioimDzRRQAyZAF4FNEWRwO+KKKYXJRBk5qaOIKelFFJiJelRyOfSiikBXaSQKSBgjmpYlZ3Vh3oopITNa2thwTWnDGoHAooqySXZjkUh9qKKYiOQ5FZOpSbUNFFUhM4vUHdpyVJqTSmlacK7YFFFNAztbOzjaME81cWCJB0FFFUIC8SelJ9pT2oopDP/2Q==', 'rocket': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAkGBwgHBgkIBwgKCgkLDRYPDQwMDRsUFRAWIB0iIiAdHx8kKDQsJCYxJx8fLT0tMTU3Ojo6Iys/RD84QzQ5Ojf/2wBDAQoKCg0MDRoPDxo3JR8lNzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzf/wAARCADIASwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDyrFLS4pcV2nORSIXXAOKqpEWJ6VdfhGPtUNsPlDHrispxTauaRk0hqQsuBkA1NCuI1+lOfBHv2xSxA+WufSqjFRdkJttahilxTsUYrSxA3FLinYpcUWAZijFPxRiiwDMUuKfijFFgGYoxT8UYpDGYpMVJikxRYRHijFPxRiiwEeKTFSYpMUARkUYp+KTFIBmKTFSYpMUDGYpMU8ikIpDGYpMU/FIRSAZSGn4pMUAMzg59KikXc4JONxOcVM3Q/SmspGCe5x+lZyim9S4tpDcAeuKAOKCSOB19TSgAcCiKtsJiGkpSKKsk0MUYp2KXFakkM/8Aqmz34qOJTt4Htj1qW54RR/ebFLCABnP3utZtXkWtIjUXHQAe3pT4h+7X6CnOABkfe7D1p0Q/dJ/uiqSs7EvUTFLingUuKoQzFGKfilxTAZijFPxRigBmKXFPxSYoAZijFPxRikAzFJipMUmKAI8UYp+KTFADCKTFSYpMUAR4oxT8UmKAGYpMU/FJikAwikxTyKQigYzFNIqQimkUgGEU0ipCKaRSGRuPlPbilc5BwOmMU7aCcHoeKQAlWOKzktSokZHA5pMDPFOJA68UmCTuP0xRbUOghpKdikqiTTxRin4pcVqSVLs4eIe+acFIVQQxXHQHpTLlv9IC+i/ln/8AXVuHnI7+vrWS1ky9kiJAQDwScVLEv7tP90Ukw8tGcdgePWpIx+7T/dFaJa2Je1wxRin4pcVZIzFGKfijFADMUuKdilxSGMxRinYoxQAzFGKdijFADMUYp+KTFADMUmKfikxQAzFJin4oxQAzFJinkUmKQDMUmKfikIoAZimkVIRTSKAGEU0ipCKaRSAYRTSKkIppFIBgHIpJG/d4ByAp/GngfMPrTACwwO4xWc0XEjxkZI69qTGAfanjBAprgtnHX1pNWGtRMUlO680lWSa+KXFOxS4rUgziA13J35H+f0qwMg4B2t2AHWobQF5C6gE5Jz+A/wATWgqqV45B9axhG+ppJ2IHAETsRk7Tz6cVMi/Iv0FMucxwSE8rtPNTqOB9K1juS9huKMU/FGKskbijFOxRigBuKKdQELd8Y5qZOyuNauwzFGKdijFMQ3FJinYoxQMbikxT8UmKAGYpMVKInaNnUcL/ADqMcjOMVN1ewWdrjcUEU7FIRTAbikIp2KCKAGUhFPxTSKAG4ppFW7+FYbp0jGFGMDOe1ViKlO6uNqzsMIppFPIpCKYhhFNIqQimkUAMxzQo2ovTcev0pcUzpx6VnNXZUWRJ8yjjOBQVwABkAUsfCfif505sY5/Oo5bq5V9bESjCgenFGKEz8wP96lqlsS9zbApk+VhkI67Tj61MBUV3xAfqP8f6VtLRMhaspWKmNAe2D/M1bDsPmUAqfzz9KisIwqYAGSB8x+nNXBbr1VRuPUkdfrWcIvlVjSTVyvcoPs8hkJLbTgHtVoDioLgBYJIyoBxxmrWK0juRLYbijFPxRirIGYoxT8UmKAG4p8Y4f/dpMVJEuUk46CoqfCy4fEQYpMU/FJiqJG4pMU/FJigY3FJin4pMUgLV1dR2OmWy+XveUM/XA4JH9KopIk8IkRSvzEEda6GJPM0e3zGhwp27hnuc1k3qlDtCAJnII9a44yvVOhr3CnikxT8UhFdZzjMUhFPxSEUAMxSEcU8imkUAWdUH+kg+san9BVIitDVB88BxjdAh/SqOKiHwlS3GEUhFPIppqhDDSEU800igBhFNQDYWI7nj15p5FM5x83YnH51nNaoqJCMAsMjJY/40m3nJbPoMdKeAA7n3H4cU7HPNSk2im9SHnfkjqP8AP86WhjhkB7k0tNEs3gKpaqT5aICRuPOPy/rWgBVG/wAtcRqvbB/mf6VpV+Bih8RJAw8vaw6kkYz+lTJcLjBySPbGaYkAeMAkscc44AqUQFRnezEdGIHFC5gdiG7VnhLSYABGFH1HereKguFBhBBP31BHb7wq1irjuyZbDMUYp+KMVZIzFJipMUmKAGYqSMfu5KbiqrmQguJiMNkIMAYz3rKq7RLpq7JsUYp+KTFaEjMUYp2KMUgGEUmKfikxSGdLpTt/Y0QTynWMEE7eckk4/DNZetMHSMs0YcsTtVMHFaWhwvHpU2/cFZwy5HGCo6Vn65aOvkXH8BTA+u5v8K82CtWXqdb1gzGxQRTyKTFekchGRSYp5FIRQAwikxT6a3Q0gLepDMdmfWBaoYqxdZX7M28uGj2lWPC8npUJFZ03dFzWow0hFPNIRWhJGRSEU800ikAwiowMsxHripTSIdokx69fyqJ36FRIMYlcA5GB/WmncDhVz/SpJQCycY4PSjA96hXG7EDYCkk85B5p2KeyA8HjIpg5APtTiwZ0YFZV3ITeOMfKpxn8AP6mtgCsiPD3jEFuSDx7kn/CqrPZIVLqzRgI2kr90dh2qc4AySAKrtHJxtUn/Z3c/wA+KdHFt+9n6E5xVqTWhLSeoy4G/YyAhfMXJPQ89qt4qG5DnygQMeavIqziqjuyZbIZijFSYpMVZIzFJipMUmKAIzwM+lUvN2wqhQHPLHdyPbFXpuIn4zweKpvaxKodRg57E4rGsm1oa02kWsU3FSkUmK1MyPFJipMUhFAxmKTFPxRikM6fSQ76O6jA24x/3yKr+IVH9lWh+XdtGfX7z1d0J1bTGQEbiQcfRR/jVTxB5Z0yHBXzAwUjvgFj/WvLi/3y9Tra9xnLkUhFSEU0ivTOQjIpCKeRSEUgGYpj/dNSkUxx8ppPYa3Eu5g6W6bMeWCM5+8d2aaRTXjX7xyTnPWn4rOmmrplT1sMIpCKeRTSK0JGGmmnmkIoAjNIzYwMds5p5FCKC4LdMdPpmon0ZUSGQY259SP0phqWXoGzkbh+HWoyu4cgc1mmymhhyfu9PWmqNuVHQHFSMpIAHGO1QSZRvlBORmjms7sdrqyOonby4JHzjapOfwrN0xfMleSMjG4gH1AAH+NXtRB+yOqjlsD8M8/pmqmlbxHvVWBYZ5GRySaub/eJEQXutmogGPlGPXPXNOKA9fzqv9obdwqFh1AfqKlVnlGTujXvjk1qpp6EOL3IrjImt4yc5lBGPTB61bxVeQRiW2SPH+sJOOv3T1q3iiG7CXQZijFSYpNtaEEeKMVJikxQBXn4Tpn5h/OoXj2oCT1OTVmfjZwc5PT6GoJo2RVUnIJ/LispmkSYjmkxUhXmjbWhBFikxUm2grSHYixSYqUikxSGdf4eto202WUoC+VAb22LWJr4UEhSATgso+rc10vhhQ2iyMO7j9EUVzHiLAviueSg4/E15dP+Odkv4ZjEU0ipSKaRXpnIREU0ipSKaRQBGRTGHFSkU1uASaT2BETjj8RQOQDQ4OPY0o5HFRHcp7DSKaRTyKaRVkjCKaaeaaaQxhFL0QnIGRt/Wg0AZAGM5/Oon0KiQygeUT/tDH50hHcVNcx7YmHcLyM9+tRDrioU1dspxZGcn+ntSMc9B+tSbDyewpmMdxSfLIFdG1rDbYAfQM35KcfqRRpkP7naQCQApHQDAxye9V9cbdcRwh8DCgj1y3+ANaVkUeFNrBZDkj/PpVLWoybWgSi3jOC6KxHTK9PpSmLbyoBx7VMhycEYYdRTWlGdsYMjei9B9TW2ljPUrSspurQKOrMfphf/AK9XNtVJEb+0rVnI3FX4Xp0H59avgUQe45LYZto21JtoxV3JsR7aTbUu2k20XCxTuhllUNg4OOOvSop1fKq+DlutWZgfOyOoAA/Mn+lQXHmEpuXBDcjP+eKxm9zSJYI5pNtSkUhFa3IIitJipSKaRSuMiIoxTyKTFAHceD4gPDhYDlpXJ965TxOgGsqcDPldfxNdt4Rjx4Yj93c/rXH+LFxrK/8AXL+pryaT/fnZP+GYpFNIxUhFIRxXq3OSxERTSKkxTWFK4EZFRuODUppjjINJ7AtyGQfLxSRjC496e/Tk/jTUBGc9Sc1C+It7CGmmnmmmtCBhppp5phpANNLFw6k/c53fT0pDSpnY+CO3BHWonsXHcilJkjkDHqDj61EGLIhIHKg1YVcRu55z8o/H/wCt/Oq8PESAgkYqNOYfQAcHIyD6inERk5JIP+yMimlCT8o3A9CKeqR4+d8N6AUSdtUEV3LV/wDvdU442sRnPXavH6tWn5YESqp8zgAAkkZ/z6VkxgXl8SgBAJJ3DglnP9BXR24jAIUYdeCD1H/1vpxShrfzCWiRTWymPLkBOwUncP15q1AD9xXVfTAyD/n0qyKZJCsnPKt/eFapJbEXb3KzK/8AatuHYHETkYGO61fxxVCHzJNVUthtkBG5ejZYVpMuG2+nFEJLUJLYaBRtp2MUuMiruTYZijFONLj5c0XHYotgTu+CSDjp7dvzqKZy7xkKw5xyPcVJG6CaQs3IY/8AAecf0ondTKm0q2Tjg+4rKUtC0tScim4qUjimGtbkWGEU0inmmmi4WGEUKpZsDGfc4pSKSlcLHpPhJQPDMIClSC+eQcnPWuM8YrjVo+DzH/U13XhVR/wjUG3odx61xPjHH9pqMc+Xx/30a8mk37dHbNe4c+aaacaaa9W5yWENRtUjUw0XCxGaY/SpDTDSb0BLUif7pGOcUxT8zZqc5PWoQfnx7VF/eKtoBphp5phNXcmw00w04mmE0XAaakgXckwJwAmT9MioiamsXUTMr4KMh3A9wBnH6VFR2iOK1C4fySsasRgZbjkE47/lVJG6hOgJ5x71NI+6RmPzOxyT2yagUYeVAQcP+fArPl1Rd9CdbgDAKlsHPbn9KkHluA0fkAdxKQCD+VUzwfcUvmHuFPuRmq5EtieYnslkN4zEkBSE4J/hUDrj3NbiXKhB5iuGGSHQZIrI0u8hh+Zoi+dx6DuT/TFaQubWfBnwFPRFTH5nrUQvbQuW5bW/BwoRmfsQDtPvmpUdGOZ5kP8AsZwv69aqiew+7GUTPYpgGp1nji+WUIB2firu+pNl0Ol8JjSW1eSTU3Xy0twQqtnPzcfSqupmE3Uotvli3HC45/E96y4riJFdgyAOQM5HOP8A9dK97FjJlU/jWNKDU5TvuaTleKjYnpyHB56VRa66kSRgY+XnOaVZrlwNgh/4E2M/nXQ5JbmSj2L0iFWwfqKfIuwKp64ya09G0bUtVijje22PG4JbqHjJ5wR3H9ar63CLSSdroqJTkiBGyyjtn0rBYmDly31NvYNK5jwJkAvgnaCB9eaZKFE8YHp0/wCBLS+aZY2DQmNE6cn+dU0kYyFwoxGM5B68jFU5e6So6mm3XHaozUT3KgDcwVu4IpVkyM7lI9jW1zOw4001FNcCNckZ+hqJb2Nu+PqadxWLJNIoJIA71EsquPlI/A9KljJIZgpO1Ccg9vWs5zsVGNz1Pw0ix6EsKgBosq+O54P49a4jxku7U48DkIef+BGu40DI02bfIHYufmxjgBRXD+M8JqSsdwJXA568mvLpu81Y7JLQ55lAzzzTVGTV5kjePcq8jIbnoRVeBAVkfI+XA/E16MayaOd02mV5OtR59aV3Useaj39eK0UiOVisKZUoZJI8q3zDtUarvJwR8oyaHLRgo6jSfm57fpULcNUhbLAD1qJ+Tk8E1LeqH0GmmGnE1G5xWlzOwhNMJoZ8dajLj1oCwpPtRDl5AORwTx7A1ExbrUmnuftsYwT1GPwqKj91lRWpPbRPKzSRRs5UZAUZOe3+P4VVMLpNIjqVIwSGBH+elSvcSqIoRlmI3uAOcnoPwGPzNOjupbeYsk0js6AfLIQOD0OOuM+tZtyTuWkrC29q1xhS8QGceYXAA+v+c1ZOkvESskRfnht+Mj6DNVxqc6bh5jOXOWYNjn2P9acmoTgfJqlxGP7rFiR+VDcxWRRtzsTAxxgYqTzT1FVkdgvQEHmnb++0D8apOyBq5YEmepqQSnGM8VV3+gpVbnvVc1xWNCR8W8A9dzcfXH9KVW3LlR0qGQOREApOIx/Mn+tS2YcTKHGATg5qIO0fv/MctxUYswAGSasrjzAJX2IOvGT+Va9t4dukmml8o+SkYdW+px+mD+VZE1hcKTJOpjU8jdwSKXOpbAtDt/CXixNFtbmS1gAhjCg72y0jE9z24zwKy/FGrrf6jJN5UBim+eNgu04PYkd+3Nc20xisZIk7spPv1/wqBJZJYxGc5U5X2rlVBKTn/XQ6faXVjotJtri6uk8uN5UY/Mqtnit3VPCVxYaY0p2RxSy7jLI2AEA4Hvkk9PSud8OzQ2dyksuJJRkqMkKvHqOT9K9H8U+JrO60cwxbjcwqM5AI468Z/wA4rjxNWaqWT/4OpvTirLQ8ou5BCdkTM4H8bcZ+gqkZnPU1YvZ4ppGbaAT6f4VRYE525xXrQempxTWug/zT60bix4AqLY+M7Tj1pYY5JX2RI0jf3UUk1pzGdixaSbbqJicKJBn860UdfMvSGcAKwIAAA57c1Tt9MvJJQDa3CjrnyzWo+lXZjkkSJw0kZDLsOdxIyf0rmq1YKW5tCnK2x6bocoi0svOQv7xo9w6HO0D9a5j4gW6tf2+JApZBwe5ya1dLvov+EeRLp2t5hcg7WOGwWAzWP4636le2osY3mCL8zoMgc9K86nK0t7HVKLa2Kuh6dLFfus8UvkTbgz7cqPQ5HpWnfeFrm0smjCYclpC5OABwq5P/AH0af4cjfS5PNlkkMjOcIhIUA/3j/hXca5rNpLpUo2ebJFzgDB44JH0OfyrnniJc7al2+ZryWSVv+AeMXlnHY8Od7epGB+HesqaRWPBwK2tY+x3MzSQTybmPImHP5isSS2fPC59xXsUZNxvLc46is7ImsIZpJ18hRKc8qvJI+ldBdeGLq1snuI43Mcx+R24ATGTknpyQPwrn7QR2sgkmjZyOQinA/E/4V3MviafVdJh0uVYiyx70Uplc8nb6/dI568H1rKvOopLl2KpxjbU4YtHG+wfvGHVgeKpzSnzCoGAK3TpxWK5nlh8vYduOSuev5YrFlt3Zy45HB3YwDW6kjFopPIaYXPfNOk+Q4zk1A7Ma6bmNhzP70zfUZJph+tFwJjKcYqzpbZ1K1AxkyqPm6daoZqfT5Nl9bt/dlU/rUz1ixx3LV6H8yRIQpXOSE+8fc9z/ACqkx2bQeuPmH1//AFVdmQNezxOyjy5mKknBHzdDTJI0kZDNdwOTJhsZVl68klen51lzaF2KhPG5eR39qaXq+2mukkfk3UD+ZnZsbJOPbFW4oNPCnfAGOfvSTbCf+A5GP1+tU6iJ5WZixkoOMcccUC2cnO40UU5aAgZShwcUqtj+lFFPpcXWxd84rKAGOQq/yFb2ivLPcRo9rbyR8b2n+VQPr/8Aroornv7iLlue0R3+h2+ghUZSVi3BUGWJ2np+Cn3/ABNeL69PDfXbSxX6yqxyFZSmPwoorOM2ndDSuZbpiA8K2XHQ+x/xqAsUYcBaKK2i+ZajasXLGYfaVYgfKpPB9jWjbpf3EpkgtppSAFYBSeTkkH86KK48XVdNXSOmhHmdiK50jUYyzNp0+zthM4/KsoxvIMxxOR6hciiilhMXOrC8kgrUYxlZDkhujwsUhH+4a0vC+9NXIMZDiNvlIwe1FFb1JuVOV+xnCKU16nWzTSQRvJLA21AScMv+NK87pD5rQNsAz1Xp+dFFeQtbHosWTzHXD2xIyDyw7c+vtUiyS4yIGAP+0v8AjRRUKV2Oxc+y3sUEd5JbMIGIKtvU5/DNZ99qckaSZjcbY8tkjGCeaKK3jCLmkzHnfK2cRqD4mZkOEJ4qobhgMA0UV7NPWJwT3HxXB3DJzXU+G5Iby+jV7RnYPkPEcFeOpzx+tFFKqtCYnqeraTo8milyYWwm8ZfCseOpH5fSvFvEn2jzzviVIx91Y8FQPwoorKL95BbQ5yUgMRioS49KKK7VsZMjJBphxnoKKKYgwvoKsvGsaWsiqFyp3EeoYjP5YooqJFxNC5spZrrUIIUMsjyhg+3qd3T/AMez+VVGsbjcR9ncyg4LNgAkc/5zRRXJ7RrQ3UE9Svc291BLi6hfzCMgNzxU0balKoYJ5gHAMiKT/wCPc0UVp7R8ilYn2a5rH//Z'}
photos = {name: np.asarray(Image.open(io.BytesIO(base64.b64decode(value)))).astype(float)/255
          for name,value in PHOTO_DATA.items()}
def u8(x):
    return np.rint(np.clip(x,0,255)).astype(np.uint8)
def gray(rgb):
    # Decode sRGB, form linear luminance, then encode for display.
    linear=np.where(rgb<=0.04045,rgb/12.92,((rgb+0.055)/1.055)**2.4)
    y=linear @ np.array([0.2126,0.7152,0.0722])
    encoded=np.where(y<=0.0031308,12.92*y,1.055*y**(1/2.4)-0.055)
    return u8(255*encoded)
def show(images,titles):
    fig,axes=plt.subplots(1,len(images),figsize=(4*len(images),3.5),squeeze=False)
    for ax,im,title in zip(axes[0],images,titles):
        ax.imshow(im,cmap='gray',vmin=0,vmax=255) if im.ndim==2 else ax.imshow(im)
        ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()
def histograms(images,titles):
    for im,title in zip(images,titles):
        plt.plot(np.bincount(im.ravel(),minlength=256),label=title)
    plt.xlabel('8-bit intensity');plt.ylabel('Pixel count');plt.legend();plt.tight_layout();plt.show()
def equalise(im):
    # Lecture convention: round(255 * CDF), without subtracting CDF_min.
    counts=np.bincount(im.ravel(),minlength=256)
    lut=u8(255*np.cumsum(counts)/counts.sum())
    return lut[im]
def endpoint_percent(im):
    return 100*np.mean((im==0)|(im==255))
def compare(im,result):
    show([im,result],['Input','Your result'])
    print('Endpoint pixels (%), input / result:',round(endpoint_percent(im),2),round(endpoint_percent(result),2))
demo_rgb=photos['astronaut'];demo_gray=gray(demo_rgb)
demo_flat=u8(90+demo_gray.astype(float)*0.25)
print('Ready. Images use RGB in [0,1]; grayscale arrays use uint8 in [0,255].')

### Enter your ID, then run
Use your own numeric ID, keeping any leading zeros. This cell generates your assignment locally. Keep the printed assignment card and image names in your submitted notebook.

In [ ]:
STUDENT_ID = ""  # Enter your student number, e.g. "20261234"
assert STUDENT_ID.isdigit() and 6 <= len(STUDENT_ID) <= 12, "Enter your numeric student ID (6–12 digits)."
seed=int.from_bytes(hashlib.sha256(STUDENT_ID.encode()).digest()[:8],'big')
rng=np.random.default_rng(seed)
names=np.array(['coffee','chelsea','rocket']);chosen=rng.permutation(names)[:2]
student_rgb=photos[chosen[0]]; second_rgb=photos[chosen[1]]
student_gray=gray(student_rgb); second_gray=gray(second_rgb)
P=dict(E=round(float(rng.uniform(.6,1.4)),2),rho=round(float(rng.uniform(.2,.7)),2),
       Z=int(rng.integers(1200,2601)),gain=round(float(rng.uniform(1.5,3)),2),
       levels=int(rng.choice([4,8,16])),step=int(rng.choice([4,6,8])),
       alpha=round(float(rng.uniform(1.2,1.8)),2),beta=int(rng.integers(-35,-9)),
       gamma=round(float(rng.uniform(.4,.8)),2),clip=round(float(rng.uniform(1.5,4)),2),
       bins=int(rng.choice([4,8,16])),low=int(rng.integers(70,91)),high=int(rng.integers(151,181)))
counts4=rng.integers(1,9,4)
cast_gains=np.array([1.25,1.0,.75])
student_cast=np.clip(student_rgb*cast_gains,0,1)
student_flat=u8(90+student_gray.astype(float)*.25)
student_noisy_flat=u8(student_flat.astype(float)+rng.normal(0,2,student_gray.shape))
student_dark=u8(student_gray.astype(float)*.45)
second_dark=u8(second_gray.astype(float)*.45)
student_hsv=rgb_to_hsv(student_rgb)
# Select a hue interval around a saturated pixel in your assigned photograph.
valid=student_hsv[...,1]>.35
centre=float(np.median(student_hsv[...,0][valid])*360) if valid.any() else 30.
hue_low=max(0,centre-25);hue_high=min(360,centre+25)
print('Assignment:',hashlib.sha256(STUDENT_ID.encode()).hexdigest()[:10])
print('Primary / transfer image:',*chosen);print('Settings:',P)
print('Four-bin counts:',counts4,'; hue interval (degrees):',round(hue_low,1),round(hue_high,1))
show([student_rgb,second_rgb],['Your primary image: '+chosen[0],'Your transfer image: '+chosen[1]])

**Setup check:** Which image is your primary image? Which is the transfer image?

Answer: …

## 1 · From light to stored numbers
### Run the example
For a fixed viewing geometry, use the simplified model **reflected light = illumination × reflectance**. It is a relative model, not a calibrated camera simulator.

For the pinhole model, all distances below are in millimetres: `u = f*X/Z`. For the sensor, use a normalised signal and an 8-bit converter: `code = round(255 × clip(gain*signal + noise, 0, 1))`.

In [ ]:
E,rho=1.0,.4
f,X,Z=35,100,2000
signal,gain,noise=.2,2,.01
print('Reflected light:',E*rho)
print('Image coordinate u (mm):',f*X/Z)
print('Sensor code:',u8(255*(gain*signal+noise)))
# Same stored square, different surroundings.
patch1=np.full((80,80),40,dtype=np.uint8);patch2=np.full((80,80),210,dtype=np.uint8)
patch1[25:55,25:55]=128;patch2[25:55,25:55]=128
show([patch1,patch2],['Centre = 128','Centre = 128'])

**Observe:** Do the centres appear equally bright? Does perception prove their stored codes differ?

Answer: …

### Your turn · Predict, then complete three expressions
Use `P['E']`, `P['rho']`, `P['Z']`, and `P['gain']`; retain `f=35`, `X=100`, signal `0.2`, noise `0.01`. Predict what doubling distance does to `u`.

In [ ]:
reflected = None  # TODO: illumination times reflectance
u = None          # TODO: pinhole image coordinate with assigned Z
sensor_code = None # TODO: assigned gain, then convert with u8
if any(x is None for x in [reflected,u,sensor_code]):
    print('Complete the three expressions above.')
else:
    print('Reflected light / u / sensor code:',reflected,u,sensor_code)
    print('At twice the distance, u =',u/2)
# Sampling changes the spatial grid; quantisation changes available levels.
coarse=student_gray[::P['step'],::P['step']]
coarse=cv2.resize(coarse,(student_gray.shape[1],student_gray.shape[0]),interpolation=cv2.INTER_NEAREST)
quantised=u8(np.rint(student_gray.astype(float)/255*(P['levels']-1))*255/(P['levels']-1))
show([student_gray,coarse,quantised],['Original','Coarser sampling',str(P['levels'])+' levels'])

**Explain (one sentence each):**
1. What changed when distance doubled: object identity or its image size?
2. Point to one detail affected by sampling and one affected by quantisation.
3. For a moving subject in low light, what is a disadvantage of a longer shutter time? How does widening the aperture differ from increasing electronic gain? Mention photons and one possible focus/background trade-off.

Answers: …

## 2 · Colour is a representation
### Run the example
RGB stores three channel values. HSV separates hue, saturation and value; it does not add information. Here hue is shown in **degrees**, saturation/value in **[0,1]**. This differs from OpenCV's 8-bit HSV ranges. Lab is another colour space, designed to make colour differences more perceptually meaningful.

The provided grayscale helper decodes sRGB, combines **linear** channels with weights 0.2126, 0.7152, 0.0722, then encodes for display.

In [ ]:
pixel=np.array([[[.8,.5,.2]]])
hsv=rgb_to_hsv(pixel)[0,0]
print('Example HSV:',hsv*np.array([360,1,1]))
h=rgb_to_hsv(demo_rgb)
demo_mask=(h[...,0]*360>=10)&(h[...,0]*360<=50)&(h[...,1]>.35)
show([demo_rgb,demo_gray,demo_mask.astype(np.uint8)*255],['RGB','Displayed luminance','Orange-range mask'])
demo_cast=np.clip(demo_rgb*cast_gains,0,1)
demo_corrected=np.clip(demo_cast/cast_gains,0,1)
show([demo_rgb,demo_cast,demo_corrected],['Original','Known colour cast','Divide by known gains'])

**Observe:** Why is `V` not the same as luminance? Which bright details fail to return perfectly after dividing by the gains?

Answer: …

### Your turn · Two lines
Create a hue mask on `student_hsv` using your printed `hue_low`, `hue_high`, and saturation > 0.35. Then correct `student_cast` by dividing by `cast_gains` and clipping to [0,1].

In [ ]:
mask = None       # TODO: combine the three comparisons with &; put each in parentheses
corrected = None  # TODO: divide by known gains and clip
if mask is None or corrected is None:
    print('Complete both lines above.')
else:
    show([student_rgb,mask.astype(np.uint8)*255,corrected],['Your image','Hue mask','Corrected cast'])
    print('Selected pixels (%):',round(100*mask.mean(),2))
    print('RGB mean absolute difference after correction:',round(float(np.mean(abs(corrected-student_rgb))),4))

**Experiment:** Remove the saturation condition and run again, then restore it. Describe which extra pixels appear and why hue is unreliable near gray. Is this mask an object detector? Why can white balance not recover clipped channels?

Answer: …

## 3 · A histogram counts; it does not locate
### Run the example
A 256-bin histogram counts each intensity. A normalised histogram divides these counts by the number of pixels. Neither records pixel positions.

In [ ]:
small=np.array([[0,0,1],[1,2,3]],dtype=np.uint8)
print('Four-bin counts:',np.bincount(small.ravel(),minlength=4))
shuffled=np.random.default_rng(7).permutation(demo_gray.ravel()).reshape(demo_gray.shape)
show([demo_gray,shuffled],['Original layout','Shuffled layout'])
histograms([demo_gray,shuffled],['Original','Shuffled (overlaps)'])
print('Histograms identical:',np.array_equal(np.bincount(demo_gray.ravel(),minlength=256),np.bincount(shuffled.ravel(),minlength=256)))

**Observe:** What useful information disappeared even though every histogram count stayed the same?

Answer: …

### Your turn · Two lines
Compute the 256-bin histogram of `student_gray`. Compute the number of bins in a 3D RGB histogram with `P['bins']` bins **per channel**. The provided code builds that RGB histogram for you.

In [ ]:
counts = None      # TODO: np.bincount on the flattened grayscale image, minlength=256
colour_bins = None # TODO: bins per channel raised to the correct power
if counts is None or colour_bins is None:
    print('Complete both lines above.')
else:
    plt.bar(np.arange(256),counts,width=1);plt.xlabel('Intensity');plt.ylabel('Count');plt.show()
    rgb_counts,_=np.histogramdd(student_rgb.reshape(-1,3),bins=(P['bins'],)*3,range=((0,1),)*3)
    print('Pixel count / histogram sum:',student_gray.size,counts.sum())
    print('RGB bins / occupied bins:',colour_bins,np.count_nonzero(rgb_counts))

**Explain:** Describe your histogram as dark/bright and narrow/broad, using a visible region as evidence. Does a broad histogram guarantee a useful image? If the number of bins per channel doubles, by what factor does total RGB bin count grow?

Answer: …

## 4 · Brightness and contrast: two different mappings
### Run the example
Use floating-point arithmetic before clipping. An affine mapping is `J = clip(alpha*I + beta, 0, 255)`; gamma is `J = 255*(I/255)**gamma`. These are digital edits, not a physical simulation of exposure. Gamma here is an intentional enhancement, separate from the sRGB encoding step.

In [ ]:
I=demo_gray.astype(float)
affine_demo=u8(1.4*I-20)
gamma_demo=u8(255*(I/255)**.6)
show([demo_gray,affine_demo,gamma_demo],['Original','alpha=1.4, beta=-20','gamma=0.6'])
histograms([demo_gray,affine_demo,gamma_demo],['Original','Affine','Gamma'])
print('Affine mapping of 100:',u8(1.4*100-20))

**Predict:** Will gamma below 1 brighten or darken middle intensities? Does adding a constant alone increase the difference between two unclipped intensities?

Answer: …

### Your turn · Two lines
Apply your assigned alpha/beta to `student_gray`; apply your assigned gamma to `student_dark`. Use `.astype(float)`, the formulas above, and `u8`.

In [ ]:
affine_result = None # TODO: assigned affine transform of student_gray
gamma_result = None  # TODO: assigned gamma transform of student_dark
if affine_result is None or gamma_result is None:
    print('Complete both lines above.')
else:
    compare(student_gray,affine_result)
    compare(student_dark,gamma_result)
    histograms([student_dark,gamma_result],['Dim input','Gamma result'])

**Experiment:** Try gamma 1.4 on the same dim image; then restore your assigned gamma. Name one region whose visibility changes. Explain why clipping two different bright values to 255 makes exact recovery impossible.

**Metric caution:** The printed endpoint percentage counts values at 0 or 255. Some were already there; it is a warning sign, not proof that every endpoint pixel lost information.

Answer: …

## 5 · Contrast stretching, CDF, and local contrast
### Run the example
Stretching maps chosen low/high intensities to 0/255. A CDF is the cumulative fraction of pixels at or below each intensity. We follow the slides' equalisation rule: **round(255 × CDF)**. Some libraries subtract the first nonzero CDF, so their outputs can differ.

In [ ]:
demo_counts=np.array([1,3,4,2])
cdf=np.cumsum(demo_counts)/demo_counts.sum()
print('CDF:',cdf,'; mapped codes:',u8(255*cdf))
stretched=u8(255*(demo_flat.astype(float)-90)/(154-90))
global_eq=equalise(demo_flat)
local_eq=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(4,4)).apply(demo_flat)
show([demo_flat,stretched,global_eq,local_eq],['Low contrast','Stretch','Global equalisation','CLAHE'])

**Observe:** Why must a CDF end at 1? Why can equalisation move existing levels but not recreate detail that was never recorded?

Answer: …

### Your turn · Four short lines
1. Stretch `student_flat` using your assigned `P['low']` and `P['high']`.
2. Calculate the CDF for your `counts4` (not the photograph).
3. Use `equalise(student_noisy_flat)`.
4. Use CLAHE on the **same noisy image**, with assigned `P['clip']` and a 4×4 tile grid.

CLAHE works locally and limits histogram peaks before blending tiles. A larger clip limit allows stronger local enhancement; it can also reveal more noise.

In [ ]:
stretch_result = None # TODO: assigned low/high stretch, followed by u8
my_cdf = None         # TODO: cumulative counts divided by total counts
eq_result = None      # TODO: supplied global equalise helper
clahe_result = None   # TODO: createCLAHE(...).apply(student_noisy_flat)
if any(x is None for x in [stretch_result,my_cdf,eq_result,clahe_result]):
    print('Complete the four lines above.')
else:
    compare(student_flat,stretch_result)
    print('Your counts / CDF / mapped codes:',counts4,my_cdf,u8(255*my_cdf))
    show([student_noisy_flat,eq_result,clahe_result],['Same noisy input','Global equalisation','CLAHE'])

**Experiment:** Change only the CLAHE clip limit to 8.0, then restore the assigned value. Inspect one smooth region and one textured region. Which setting would you keep for seeing useful detail, and what unwanted effect supports your decision? Is higher contrast always better?

Answer: …

## 6 · Independent task: does your decision transfer?
No new method is needed. Use your assigned **second photograph**, not the example image. You may discuss concepts with classmates, but write your own code, observations and conclusion.

**Before running:** Choose either gamma correction or CLAHE for the dim image. Predict which visible detail will improve, and one possible failure. Use the same chosen method and parameters on both primary and transfer images so the comparison is fair.

Prediction and chosen method: …

### Write two similar lines
Set `primary_fixed` and `transfer_fixed` using `student_dark` and `second_dark`. Adapt one of the earlier examples. Gamma uses `P['gamma']`; CLAHE uses `P['clip']` and a 4×4 grid. These arrays are prepared in setup, so this task does not depend on your earlier answers.

In [ ]:
primary_fixed = None  # TODO: chosen method on student_dark
transfer_fixed = None # TODO: same method/settings on second_dark
if primary_fixed is None or transfer_fixed is None:
    print('Write your two transformation lines above.')
else:
    show([student_dark,primary_fixed,second_dark,transfer_fixed],['Primary input','Primary output','Transfer input','Transfer output'])
    for name,original,changed in [('Primary',student_dark,primary_fixed),('Transfer',second_dark,transfer_fixed)]:
        print(name,'input/output endpoint %:',round(endpoint_percent(original),2),round(endpoint_percent(changed),2))

### Your evidence and conclusion
Fill the table with your actual results. Name a visible region rather than saying only “better”.

| Image | Input → output endpoint % | Detail improved | Detail/noise made worse, or none observed |
|---|---|---|---|
| Primary | … | … | … |
| Transfer | … | … | … |

**Conclusion (4–6 sentences):** Did your prediction hold on both images? Explain the mapping you used and why it helped or failed. Does the endpoint metric agree with your visual judgement? State one limitation and one different method you would try next. Do not claim that appearance alone proves better recognition accuracy.

Answer: …

## Next lecture · A five-minute look at neighbours
Run this complete cell. Today's global intensity mappings transform values without examining neighbouring pixel arrangements. Next week, filtering and gradients will use those neighbours. No convolution implementation is required today.

In [ ]:
smooth=cv2.GaussianBlur(student_gray,(5,5),1.2)
gx=cv2.Sobel(smooth,cv2.CV_32F,1,0,ksize=3)
gy=cv2.Sobel(smooth,cv2.CV_32F,0,1,ksize=3)
magnitude=np.hypot(gx,gy)
edge_view=u8(255*magnitude/max(float(magnitude.max()),1))
show([student_gray,smooth,edge_view],['Original','Neighbourhood smoothing','Gradient magnitude (scaled)'])

**One insight:** Which fine detail becomes weaker after smoothing? Why would shuffling pixels change this edge image even though it preserves the histogram?

Answer: …

## Before submitting
- Use your own ID; keep the assignment card and your two dataset image names visible.
- Complete each TODO and answer cell, including experiment observations and the transfer table.
- Restore assigned values after the requested comparisons. Re-run from the top and keep outputs visible.
- Follow the course submission rule: `StudentID_Lab2_CV.ipynb` and `StudentID_Lab2_CV_Report.pdf`. The short PDF should contain your transfer comparison and conclusion; submit by the deadline shown on the course page.

**What matters:** correct small adaptations, fair comparisons, and conclusions supported by your images. There is no single “best-looking” result for every image.

### Dataset and references
Included JPEG thumbnails are derived from the [scikit-image example dataset](https://scikit-image.org/docs/stable/api/skimage.data): astronaut (NASA), coffee (Rachel Michetti), Chelsea the cat (Stefan van der Walt), and rocket (SpaceX). The dataset documentation records their public-domain/no-known-copyright-restriction status. Thumbnails are embedded to make assignments reproducible without downloads.

Methods: Session 2 course slides; [OpenCV equalisation and CLAHE](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html); [OpenCV filtering](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html). Our global equalisation helper deliberately uses the slide CDF convention stated above.